In [1]:
import itertools
import pandas as pd
import json
from pathlib import Path
import numpy as np
from typing import List, Tuple

In [2]:
# -----------------------------
# 1) Configure methods & tasks
# -----------------------------

# Extended mapping from directory names to display names
MODEL_NAME_MAPPING = {
    # Naive baselines
    "naive_baseline_all_full": "Naive (Full Box)",
    "naive_baseline_all_random": "Naive (Random Box)",
    
    # Commercial LVLMs
    "gpt-4.1": "GPT-4.1",
    "gemini-2.0-flash": "Gemini-2.0-Flash",
    "claude-sonnet-4-20250514": "Claude-Sonnet-4",
    
    # Open-source LVLMs
    "llava-hf_llava-v1.6-mistral-7b-hf": "Llava-v1.6-Mistral-7B",
    "Qwen_Qwen2.5-VL-7B-Instruct": "Qwen2.5-VL-7B",
    "mistralai_Pixtral-12B-2409": "Pixtral-12B",
    
    # CLIP-based models
    "peskavlp": "PeskaVLP",
    "raso": "RASO",
    
    # Task-specific models
    "cholenet": "CholeNet",
    "gonogonet": "GoNoGoNet"
}

# Tasks
TASKS = {
    "CholecSeg8k": ["Presence", "IoU"]
}

In [3]:
# -----------------------------
# 2) Load results from JSON files with Bootstrap
# -----------------------------

import numpy as np
from pathlib import Path

# Define result directories for all three datasets
results_dirs = {
    "CholecSeg8k": Path("/shared_data0/weiqiuy/llm_cholec_organ/results/bbox_cholecseg8k_local_quick"),
    "CholecOrgans": Path("/shared_data0/weiqiuy/llm_cholec_organ/results/bbox_cholec_organs_quick"),
    "CholecGoNoGo": Path("/shared_data0/weiqiuy/llm_cholec_organ/results/bbox_cholec_gonogo_quick")
}

def bootstrap_std(data: List[float], n_bootstrap: int = 1000, seed: int = 42) -> float:
    """Compute bootstrap standard deviation of the mean."""
    np.random.seed(seed)
    data = np.array(data)
    n = len(data)
    
    if n == 0:
        return 0.0
    
    bootstrap_means = []
    for _ in range(n_bootstrap):
        # Sample with replacement
        sample = np.random.choice(data, size=n, replace=True)
        bootstrap_means.append(np.mean(sample))
    
    return np.std(bootstrap_means)

def load_summary_results(dataset="CholecSeg8k", mode="zeroshot_combined", compute_bootstrap=True):
    """Load results from individual test files and compute metrics with bootstrap confidence intervals.
    Now loads BOTH bbox-to-bbox and bbox-to-mask IoU values for all models.
    Handles corrupted/empty JSON files gracefully.
    
    Args:
        dataset: Dataset name ("CholecSeg8k", "CholecOrgans", or "CholecGoNoGo")
        mode: Evaluation mode (e.g., "zeroshot_combined")
        compute_bootstrap: If True, compute bootstrap standard deviations
    """
    if dataset not in results_dirs:
        print(f"Warning: Unknown dataset {dataset}")
        return {}
    
    results_dir = results_dirs[dataset]
    mode_dir = results_dir / mode
    results = {}
    
    if not mode_dir.exists():
        print(f"Warning: {mode_dir} does not exist")
        return results
    
    for model_dir in mode_dir.iterdir():
        if not model_dir.is_dir():
            continue
            
        model_name = model_dir.name
        if model_name not in MODEL_NAME_MAPPING:
            continue
        display_name = MODEL_NAME_MAPPING.get(model_name, model_name)
        
        # Load all test result files
        test_files = list(model_dir.glob("test_*.json"))
        
        if not test_files:
            print(f"No test files found for {model_name} in {dataset}")
            continue
        
        all_y_true = []
        all_y_pred = []
        all_ious_bbox = []  # bbox-to-bbox IoU
        all_ious_mask = []  # bbox-to-mask IoU
        all_binary_correct = []  # For bootstrap of presence accuracy
        
        # Track skipped files
        skipped_files = []
        
        for test_file in test_files:
            try:
                # Check if file is empty
                if test_file.stat().st_size == 0:
                    skipped_files.append((test_file.name, "Empty file"))
                    continue
                
                with open(test_file, 'r') as f:
                    content = f.read()
                    if not content.strip():
                        skipped_files.append((test_file.name, "Empty content"))
                        continue
                    
                    data = json.loads(content)
                
                # Collect presence accuracy data
                if 'y_true' in data and 'y_pred' in data:
                    all_y_true.extend(data['y_true'])
                    all_y_pred.extend(data['y_pred'])
                    # Store binary correctness for each prediction
                    all_binary_correct.extend([1 if yt == yp else 0 
                                              for yt, yp in zip(data['y_true'], data['y_pred'])])
                
                # Collect BOTH IoU types from individual organ data
                for organ in data.get('organs', []):
                    # Bbox-to-bbox IoU - check multiple possible key names
                    if 'iou_bbox_to_bbox' in organ and organ['iou_bbox_to_bbox'] is not None:
                        all_ious_bbox.append(organ['iou_bbox_to_bbox'])
                    elif 'iou' in organ and organ['iou'] is not None:
                        all_ious_bbox.append(organ['iou'])
                    
                    # Bbox-to-mask IoU
                    if 'iou_bbox_to_mask' in organ and organ['iou_bbox_to_mask'] is not None:
                        all_ious_mask.append(organ['iou_bbox_to_mask'])
                        
            except json.JSONDecodeError as e:
                skipped_files.append((test_file.name, f"JSON error: {str(e)[:50]}"))
                continue
            except Exception as e:
                skipped_files.append((test_file.name, f"Error: {str(e)[:50]}"))
                continue
        
        # Report skipped files if any
        if skipped_files:
            print(f"Warning: Skipped {len(skipped_files)} files for {display_name} in {dataset}:")
            for filename, reason in skipped_files[:3]:  # Show first 3
                print(f"  - {filename}: {reason}")
            if len(skipped_files) > 3:
                print(f"  ... and {len(skipped_files) - 3} more")
        
        # Compute metrics
        metrics = {}
        
        # Presence accuracy with bootstrap
        if all_binary_correct:
            metrics['presence_accuracy'] = np.mean(all_binary_correct)
            if compute_bootstrap:
                metrics['presence_accuracy_std'] = bootstrap_std(all_binary_correct)
        
        # Bbox-to-bbox IoU with bootstrap
        if all_ious_bbox:
            metrics['mean_iou_bbox'] = np.mean(all_ious_bbox)
            metrics['iou_at_50_bbox'] = np.mean([iou >= 0.5 for iou in all_ious_bbox])
            
            if compute_bootstrap:
                metrics['mean_iou_bbox_std'] = bootstrap_std(all_ious_bbox)
                iou_at_50_binary = [1 if iou >= 0.5 else 0 for iou in all_ious_bbox]
                metrics['iou_at_50_bbox_std'] = bootstrap_std(iou_at_50_binary)
        
        # Bbox-to-mask IoU with bootstrap
        if all_ious_mask:
            metrics['mean_iou_mask'] = np.mean(all_ious_mask)
            metrics['iou_at_50_mask'] = np.mean([iou >= 0.5 for iou in all_ious_mask])
            
            if compute_bootstrap:
                metrics['mean_iou_mask_std'] = bootstrap_std(all_ious_mask)
                iou_at_50_binary = [1 if iou >= 0.5 else 0 for iou in all_ious_mask]
                metrics['iou_at_50_mask_std'] = bootstrap_std(iou_at_50_binary)
        
        results[display_name] = metrics
        
        # Report successful loading
        valid_files = len(test_files) - len(skipped_files)
        print(f"Loaded {valid_files} files for {display_name} in {dataset} (both IoU types)")
    
    return results

# Load zeroshot_combined results for ALL THREE datasets with bootstrap
print("="*80)
print("Loading results for ALL THREE datasets (CholecSeg8k, CholecOrgans, CholecGoNoGo)...")
print("="*80)

all_results = {}

# Load CholecSeg8k results
print("\n1. Loading CholecSeg8k results...")
zeroshot_results = load_summary_results("CholecSeg8k", "zeroshot_combined", compute_bootstrap=True)
all_results["CholecSeg8k"] = {"zeroshot_combined": zeroshot_results}
print(f"   Loaded results for {len(zeroshot_results)} models")

# Load CholecOrgans results  
print("\n2. Loading CholecOrgans results...")
cholec_organs_results = load_summary_results("CholecOrgans", "zeroshot_combined", compute_bootstrap=True)
all_results["CholecOrgans"] = {"zeroshot_combined": cholec_organs_results}
print(f"   Loaded results for {len(cholec_organs_results)} models")

# Load CholecGoNoGo results
print("\n3. Loading CholecGoNoGo results...")
cholec_gonogo_results = load_summary_results("CholecGoNoGo", "zeroshot_combined", compute_bootstrap=True)
all_results["CholecGoNoGo"] = {"zeroshot_combined": cholec_gonogo_results}
print(f"   Loaded results for {len(cholec_gonogo_results)} models")

print("\n" + "="*80)
print(f"Total models loaded: CholecSeg8k={len(zeroshot_results)}, CholecOrgans={len(cholec_organs_results)}, CholecGoNoGo={len(cholec_gonogo_results)}")

Loading results for ALL THREE datasets (CholecSeg8k, CholecOrgans, CholecGoNoGo)...

1. Loading CholecSeg8k results...
Loaded 200 files for GPT-4.1 in CholecSeg8k (both IoU types)
Loaded 200 files for Gemini-2.0-Flash in CholecSeg8k (both IoU types)
Loaded 200 files for Naive (Random Box) in CholecSeg8k (both IoU types)
Loaded 200 files for CholeNet in CholecSeg8k (both IoU types)
Loaded 200 files for RASO in CholecSeg8k (both IoU types)
Loaded 200 files for Llava-v1.6-Mistral-7B in CholecSeg8k (both IoU types)
Loaded 200 files for GoNoGoNet in CholecSeg8k (both IoU types)
Loaded 200 files for Claude-Sonnet-4 in CholecSeg8k (both IoU types)
Loaded 200 files for Pixtral-12B in CholecSeg8k (both IoU types)
Loaded 200 files for PeskaVLP in CholecSeg8k (both IoU types)
Loaded 200 files for Naive (Full Box) in CholecSeg8k (both IoU types)
Loaded 200 files for Qwen2.5-VL-7B in CholecSeg8k (both IoU types)
   Loaded results for 12 models

2. Loading CholecOrgans results...
Loaded 140 files fo

In [4]:
# -----------------------------
# 9) Comprehensive Main Table with Model Types and Both IoU Types
# -----------------------------

print("\n" + "="*60)
print("COMPREHENSIVE MAIN TABLE - With Model Categories")
print("="*60)

# Dynamically discover all unique models from all_results
all_discovered_models = set()
for dataset_name in all_results:
    if "zeroshot_combined" in all_results[dataset_name]:
        all_discovered_models.update(all_results[dataset_name]["zeroshot_combined"].keys())

print(f"\nDiscovered {len(all_discovered_models)} unique models across all datasets:")
for model in sorted(all_discovered_models):
    print(f"  - {model}")

# Define model categories based on discovered models
MODEL_CATEGORIES = {
    "Baselines": [],
    "Commercial LVLMs": [],
    "Open-Source LVLMs": [],
    "CLIP-based Models": [],
    "Task-Specific Models": []
}

# Categorize discovered models
for model in all_discovered_models:
    model_lower = model.lower()
    
    # Baselines
    if "naive" in model_lower or "baseline" in model_lower:
        MODEL_CATEGORIES["Baselines"].append(model)
    # Commercial LVLMs
    elif any(x in model_lower for x in ["gpt", "gemini", "claude"]):
        MODEL_CATEGORIES["Commercial LVLMs"].append(model)
    # Open-source LVLMs
    elif any(x in model_lower for x in ["llava", "qwen", "pixtral"]):
        MODEL_CATEGORIES["Open-Source LVLMs"].append(model)
    # CLIP-based models
    elif any(x in model_lower for x in ["peskavlp", "raso"]):
        MODEL_CATEGORIES["CLIP-based Models"].append(model)
    # Task-specific models
    elif any(x in model_lower for x in ["cholenet", "gonogo"]):
        MODEL_CATEGORIES["Task-Specific Models"].append(model)
    else:
        # If we can't categorize, add to Task-Specific as a catch-all
        MODEL_CATEGORIES["Task-Specific Models"].append(model)

# Sort models within each category
for category in MODEL_CATEGORIES:
    MODEL_CATEGORIES[category].sort()

# Create ordered list of all models
ALL_METHODS_ORDERED = []
for category in MODEL_CATEGORIES:
    ALL_METHODS_ORDERED.extend(MODEL_CATEGORIES[category])

print("\nCategorized models:")
for category, models in MODEL_CATEGORIES.items():
    if models:
        print(f"\n{category}:")
        for model in models:
            print(f"  - {model}")

# Tasks with both IoU types for localization
TASKS_FULL = {
    "CholecOrgans": ["Presence", "IoU-B", "IoU-M"],
    "CholecGoNoGo": ["Presence", "IoU-B", "IoU-M"],
    "CholecSeg8k":  ["Presence", "IoU-B", "IoU-M"],
}

# Build the full table structure
import itertools
columns_full = pd.MultiIndex.from_tuples(
    list(itertools.chain.from_iterable(
        [[(task, metric) for metric in TASKS_FULL[task]] for task in TASKS_FULL]
    )),
    names=["Task", "Metric"]
)

# Initialize dataframe with empty strings
df_full = pd.DataFrame("", index=ALL_METHODS_ORDERED, columns=columns_full)
df_full_raw = pd.DataFrame(np.nan, index=ALL_METHODS_ORDERED, columns=columns_full)  # For numeric comparisons

# Track maximum standard deviations for reporting
max_stds = {
    'presence': 0.0,
    'iou_bbox': 0.0,
    'iou_mask': 0.0
}

# Mark N/A cells based on model capabilities
NA_STR = "—"

# CLIP-based models that don't do localization
CLIP_MODELS = ["PeskaVLP", "RASO"]

def is_applicable(method: str, task: str, metric: str) -> bool:
    """Determine if a model can perform a specific task/metric combination."""
    method_lower = method.lower()
    
    # Naive baselines can do all tasks
    if "naive" in method_lower or "baseline" in method_lower:
        return True
    
    # Commercial and Open-Source LVLMs can do all tasks
    if any(x in method_lower for x in ["gpt", "gemini", "claude", "llava", "qwen", "pixtral"]):
        return True
    
    # CLIP-based models can do presence but not IoU
    if any(x in method_lower for x in ["peskavlp", "raso"]):
        return (metric == "Presence")
    
    # Task-specific models
    if "cholenet" in method_lower:
        # CholeNet can do all metrics for CholecOrgans and partial for others
        return True  # Let's see what data we actually have
    
    if "gonogo" in method_lower:
        # GoNoGo/GoNoGoNet can do all metrics for CholecGoNoGo and partial for others
        return True  # Let's see what data we actually have
    
    return True  # Default to True and let the data determine availability

# Fill in the actual data we have from all three datasets
for dataset_name in ["CholecOrgans", "CholecGoNoGo", "CholecSeg8k"]:
    if dataset_name in all_results and "zeroshot_combined" in all_results[dataset_name]:
        results = all_results[dataset_name]["zeroshot_combined"]
        
        for model_name in ALL_METHODS_ORDERED:
            if model_name in results:
                metrics = results[model_name]
                
                # Presence - store only mean values, track max std
                if 'presence_accuracy' in metrics:
                    mean_val = metrics['presence_accuracy']
                    df_full.loc[model_name, (dataset_name, "Presence")] = f"{mean_val:.3f}"
                    df_full_raw.loc[model_name, (dataset_name, "Presence")] = mean_val
                    if 'presence_accuracy_std' in metrics:
                        max_stds['presence'] = max(max_stds['presence'], metrics['presence_accuracy_std'])
                
                # For CLIP models, set IoU to NA_STR instead of 0.000
                if model_name in CLIP_MODELS:
                    df_full.loc[model_name, (dataset_name, "IoU-B")] = NA_STR
                    df_full.loc[model_name, (dataset_name, "IoU-M")] = NA_STR
                else:
                    # Bbox-to-bbox IoU
                    if 'mean_iou_bbox' in metrics:
                        mean_val = metrics['mean_iou_bbox']
                        df_full.loc[model_name, (dataset_name, "IoU-B")] = f"{mean_val:.3f}"
                        df_full_raw.loc[model_name, (dataset_name, "IoU-B")] = mean_val
                        if 'mean_iou_bbox_std' in metrics:
                            max_stds['iou_bbox'] = max(max_stds['iou_bbox'], metrics['mean_iou_bbox_std'])
                    
                    # Bbox-to-mask IoU
                    if 'mean_iou_mask' in metrics:
                        mean_val = metrics['mean_iou_mask']
                        df_full.loc[model_name, (dataset_name, "IoU-M")] = f"{mean_val:.3f}"
                        df_full_raw.loc[model_name, (dataset_name, "IoU-M")] = mean_val
                        if 'mean_iou_mask_std' in metrics:
                            max_stds['iou_mask'] = max(max_stds['iou_mask'], metrics['mean_iou_mask_std'])

# After filling data, mark cells without data as N/A
for method in df_full.index:
    for col in df_full.columns:
        if df_full.loc[method, col] == "":
            if not is_applicable(method, col[0], col[1]):
                df_full.loc[method, col] = NA_STR

# Find best and second-best for each column (excluding CLIP models for IoU metrics)
best_second_best = {}
for col in df_full_raw.columns:
    # For IoU metrics, exclude CLIP models
    if col[1] in ["IoU-B", "IoU-M"]:
        # Exclude CLIP models from comparison
        col_values = df_full_raw[col].copy()
        for clip_model in CLIP_MODELS:
            if clip_model in col_values.index:
                col_values = col_values.drop(clip_model)
        col_values = col_values.dropna()
    else:
        col_values = df_full_raw[col].dropna()
    
    if len(col_values) >= 2:
        sorted_values = col_values.nlargest(2)
        best_second_best[col] = {
            'best': sorted_values.iloc[0],
            'second': sorted_values.iloc[1]
        }
    elif len(col_values) == 1:
        best_second_best[col] = {
            'best': col_values.iloc[0],
            'second': None
        }

# Apply formatting to best and second-best values
df_formatted = df_full.copy()
for col in df_full_raw.columns:
    if col in best_second_best:
        for idx in df_full_raw.index:
            # Skip CLIP models for IoU formatting
            if idx in CLIP_MODELS and col[1] in ["IoU-B", "IoU-M"]:
                continue
                
            if not pd.isna(df_full_raw.loc[idx, col]):
                value = df_full_raw.loc[idx, col]
                formatted_value = df_full.loc[idx, col]
                
                if value == best_second_best[col]['best']:
                    # Bold the best value
                    df_formatted.loc[idx, col] = f"\\textbf{{{formatted_value}}}"
                elif best_second_best[col]['second'] is not None and value == best_second_best[col]['second']:
                    # Italicize the second-best value
                    df_formatted.loc[idx, col] = f"\\textit{{{formatted_value}}}"

print("\n" + "="*80)
print("Data-filled table (all available models and datasets):")
print("="*80)
print(df_full.to_string())

print("\nMaximum bootstrap standard deviations:")
print(f"  Presence: {max_stds['presence']:.3f}")
print(f"  IoU-B: {max_stds['iou_bbox']:.3f}")
print(f"  IoU-M: {max_stds['iou_mask']:.3f}")

# Generate LaTeX table with category rows and midrules between categories
def generate_latex_with_category_rows(df, model_categories, max_stds):
    lines = []
    
    # Table header
    lines.append("\\begin{table*}[t]")
    lines.append("\\centering")
    lines.append("\\small")
    lines.append("\\setlength{\\tabcolsep}{4pt}")
    
    # Calculate number of columns
    n_data_cols = len(df.columns)
    col_spec = "l" + "c" * n_data_cols  # Left-aligned for method, centered for data
    
    lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
    lines.append("\\toprule")
    
    # First header row: dataset names
    first_row = ["\\multirow{2}{*}{Method}"]
    for task in TASKS_FULL:
        n = len(TASKS_FULL[task])
        first_row.append(f"\\multicolumn{{{n}}}{{c}}{{\\textbf{{{task}}}}}")
    lines.append(" & ".join(first_row) + " \\\\")
    
    # Second header row: metrics
    second_row = [""]  # Empty for Method column
    for task in TASKS_FULL:
        for metric in TASKS_FULL[task]:
            if metric == "IoU-B":
                second_row.append("IoU-B$^a$")
            elif metric == "IoU-M":
                second_row.append("IoU-M$^b$")
            else:
                second_row.append(metric)
    lines.append(" & ".join(second_row) + " \\\\")
    lines.append("\\midrule")
    
    # Table body with category rows and midrules between categories
    for cat_idx, (category, models) in enumerate(model_categories.items()):
        if not models:  # Skip empty categories
            continue
            
        # Add category header row spanning all columns
        n_cols_total = 1 + n_data_cols  # Method column + data columns
        lines.append(f"\\multicolumn{{{n_cols_total}}}{{l}}{{\\textit{{{category}}}}} \\\\")
        
        # Add each model in the category
        for model in models:
            if model in df.index:
                row_cells = []
                
                # Add model name (not in bold, just regular)
                row_cells.append(model)
                
                # Add data values (with best/second formatting)
                for col in df.columns:
                    value = str(df.loc[model, col])
                    row_cells.append(value)
                
                lines.append(" & ".join(row_cells) + " \\\\")
        
        # Add midrule between categories (except after last category)
        if cat_idx < len(model_categories) - 1:
            # Check if next category has models
            next_categories = list(model_categories.keys())[cat_idx+1:]
            has_more_models = any(model_categories[cat] for cat in next_categories)
            if has_more_models:
                lines.append("\\midrule")
    
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    
    # Caption with max standard deviations reported
    caption = ("Comparison of surgical organ detection models grouped by type. "
               "Models are dynamically discovered from available results. "
               "Naive baselines predict fixed patterns; "
               "Commercial and Open-Source LVLMs can perform all tasks; "
               "CLIP-based models support presence detection only (— indicates not applicable); "
               "Task-Specific models are trained for particular datasets. "
               "Data shown is from zero-shot combined setting. "
               "Best values are in \\textbf{bold}, second-best in \\textit{italics}. "
               f"Bootstrap standard deviations (1000 samples) were at most "
               f"{max_stds['presence']:.3f} for Presence, "
               f"{max_stds['iou_bbox']:.3f} for IoU-B, and "
               f"{max_stds['iou_mask']:.3f} for IoU-M. "
               "$^a$IoU-B: bbox-to-bbox IoU. $^b$IoU-M: bbox-to-mask IoU.")
    
    lines.append(f"\\caption{{{caption}}}")
    lines.append("\\label{tab:main_results}")
    lines.append("\\end{table*}")
    
    return "\n".join(lines)

# Generate the LaTeX code using formatted dataframe
latex_code = generate_latex_with_category_rows(df_formatted, MODEL_CATEGORIES, max_stds)

print("\n" + "="*80)
print("LATEX TABLE - WITH MIDRULES BETWEEN CATEGORIES")
print("="*80)
print(latex_code)

# Save to latex folder
import os
latex_dir = "latex"
os.makedirs(latex_dir, exist_ok=True)
latex_file = os.path.join(latex_dir, "main_table_all_models.tex")

with open(latex_file, "w") as f:
    f.write(latex_code)
print(f"\n✅ Formatted LaTeX table with all models saved to {latex_file}")

# Also create a simplified view for display
print("\n" + "="*80)
print("BEST AND SECOND-BEST PER COLUMN")
print("="*80)

for col in best_second_best:
    if col[0] in ["CholecOrgans", "CholecGoNoGo", "CholecSeg8k"]:  # Only show for columns with data
        best_val = best_second_best[col]['best']
        second_val = best_second_best[col]['second']
        
        # Find which models have these values
        best_models = df_full_raw[df_full_raw[col] == best_val].index.tolist()
        second_models = df_full_raw[df_full_raw[col] == second_val].index.tolist() if second_val else []
        
        print(f"\n{col[0]} - {col[1]}:")
        print(f"  Best ({best_val:.3f}): {', '.join(best_models)}")
        if second_models:
            print(f"  Second ({second_val:.3f}): {', '.join(second_models)}")

# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total unique models discovered: {len(all_discovered_models)}")
print(f"Models per category:")
for category, models in MODEL_CATEGORIES.items():
    if models:
        print(f"  {category}: {len(models)} models")

# Show which models have data for each dataset
print(f"\nModels with results per dataset:")
for dataset_name in ["CholecOrgans", "CholecGoNoGo", "CholecSeg8k"]:
    if dataset_name in all_results and "zeroshot_combined" in all_results[dataset_name]:
        n_models = len(all_results[dataset_name]["zeroshot_combined"])
        print(f"  {dataset_name}: {n_models} models")


COMPREHENSIVE MAIN TABLE - With Model Categories

Discovered 12 unique models across all datasets:
  - CholeNet
  - Claude-Sonnet-4
  - GPT-4.1
  - Gemini-2.0-Flash
  - GoNoGoNet
  - Llava-v1.6-Mistral-7B
  - Naive (Full Box)
  - Naive (Random Box)
  - PeskaVLP
  - Pixtral-12B
  - Qwen2.5-VL-7B
  - RASO

Categorized models:

Baselines:
  - Naive (Full Box)
  - Naive (Random Box)

Commercial LVLMs:
  - Claude-Sonnet-4
  - GPT-4.1
  - Gemini-2.0-Flash

Open-Source LVLMs:
  - Llava-v1.6-Mistral-7B
  - Pixtral-12B
  - Qwen2.5-VL-7B

CLIP-based Models:
  - PeskaVLP
  - RASO

Task-Specific Models:
  - CholeNet
  - GoNoGoNet

Data-filled table (all available models and datasets):
Task                  CholecOrgans               CholecGoNoGo               CholecSeg8k              
Metric                    Presence  IoU-B  IoU-M     Presence  IoU-B  IoU-M    Presence  IoU-B  IoU-M
Naive (Full Box)             0.998  0.411  0.185        1.000  0.221  0.107       0.423  0.295  0.161
Naive (Rand

In [5]:
# RESTART KERNEL AND RE-RUN ALL CELLS ABOVE THIS POINT
# The per-organ analysis has been updated with better error handling

print("Per-organ analysis cell has been updated.")
print("Please restart the kernel and re-run all cells to use the fixed version.")
print()
print("The fixed version includes:")
print("- Robust organ name extraction (tries multiple field names)")
print("- Better error handling for malformed JSON files")
print("- Flexible IoU field detection")
print("- Graceful degradation when data is missing")
print()
print("After restarting the kernel, the per-organ analysis should work correctly.")

Per-organ analysis cell has been updated.
Please restart the kernel and re-run all cells to use the fixed version.

The fixed version includes:
- Robust organ name extraction (tries multiple field names)
- Better error handling for malformed JSON files
- Flexible IoU field detection
- Graceful degradation when data is missing

After restarting the kernel, the per-organ analysis should work correctly.


In [6]:
# -----------------------------
# 11) Per-Organ Analysis Summary and Statistics
# -----------------------------

print("\n" + "="*80)
print("PER-ORGAN SUMMARY ANALYSIS")
print("="*80)

def create_organ_difficulty_ranking(per_organ_results, dataset_name):
    """Analyze which organs are hardest/easiest to detect across all models."""
    
    # Get all unique organs
    all_organs = set()
    for model_data in per_organ_results.values():
        all_organs.update(model_data.keys())
    
    organ_stats = {}
    
    for organ in all_organs:
        presence_scores = []
        iou_bbox_scores = []
        
        for model, organ_data in per_organ_results.items():
            if organ in organ_data:
                presence_scores.append(organ_data[organ]['presence_acc'])
                if organ_data[organ]['iou_bbox'] > 0:
                    iou_bbox_scores.append(organ_data[organ]['iou_bbox'])
        
        organ_stats[organ] = {
            'mean_presence': np.mean(presence_scores) if presence_scores else 0,
            'std_presence': np.std(presence_scores) if len(presence_scores) > 1 else 0,
            'mean_iou_bbox': np.mean(iou_bbox_scores) if iou_bbox_scores else 0,
            'std_iou_bbox': np.std(iou_bbox_scores) if len(iou_bbox_scores) > 1 else 0,
            'n_models_presence': len(presence_scores),
            'n_models_iou': len(iou_bbox_scores)
        }
    
    return organ_stats

def create_model_specialization_analysis(per_organ_results, dataset_name):
    """Analyze which models are specialists vs generalists."""
    
    model_stats = {}
    
    for model, organ_data in per_organ_results.items():
        presence_scores = [data['presence_acc'] for data in organ_data.values()]
        iou_scores = [data['iou_bbox'] for data in organ_data.values() if data['iou_bbox'] > 0]
        
        model_stats[model] = {
            'mean_presence': np.mean(presence_scores) if presence_scores else 0,
            'std_presence': np.std(presence_scores) if len(presence_scores) > 1 else 0,
            'mean_iou': np.mean(iou_scores) if iou_scores else 0,
            'std_iou': np.std(iou_scores) if len(iou_scores) > 1 else 0,
            'n_organs': len(organ_data),
            'n_organs_with_iou': len(iou_scores),
            'cv_presence': np.std(presence_scores) / np.mean(presence_scores) if presence_scores and np.mean(presence_scores) > 0 else 0,
            'cv_iou': np.std(iou_scores) / np.mean(iou_scores) if iou_scores and np.mean(iou_scores) > 0 else 0
        }
    
    return model_stats

def print_organ_difficulty_ranking(organ_stats, dataset_name, metric='mean_presence'):
    """Print organs ranked by difficulty."""
    
    print(f"\n{dataset_name} - Organ Difficulty Ranking (by {metric.replace('_', ' ').title()}):")
    print("=" * 80)
    
    # Sort organs by the metric (ascending for difficulty, descending for performance)
    sorted_organs = sorted(organ_stats.items(), key=lambda x: x[1][metric], reverse=(metric.startswith('mean')))
    
    print(f"{'Rank':<4} {'Organ':<25} {'Mean':<8} {'Std':<8} {'Models':<8}")
    print("-" * 60)
    
    for i, (organ, stats) in enumerate(sorted_organs, 1):
        mean_val = stats[metric]
        std_val = stats[metric.replace('mean', 'std')]
        n_models = stats['n_models_presence'] if 'presence' in metric else stats['n_models_iou']
        
        print(f"{i:<4} {organ[:24]:<25} {mean_val:<8.3f} {std_val:<8.3f} {n_models:<8}")

def print_model_specialization(model_stats, dataset_name):
    """Print model specialization analysis."""
    
    print(f"\n{dataset_name} - Model Specialization Analysis:")
    print("=" * 80)
    print("Lower CV (Coefficient of Variation) indicates more consistent performance across organs")
    print()
    
    # Sort by consistency (lower CV is better)
    sorted_models = sorted(model_stats.items(), key=lambda x: x[1]['cv_presence'])
    
    print(f"{'Model':<20} {'Mean Pres':<10} {'CV Pres':<8} {'Mean IoU':<10} {'CV IoU':<8} {'Organs':<8}")
    print("-" * 80)
    
    for model, stats in sorted_models:
        print(f"{model[:19]:<20} {stats['mean_presence']:<10.3f} {stats['cv_presence']:<8.3f} "
              f"{stats['mean_iou']:<10.3f} {stats['cv_iou']:<8.3f} {stats['n_organs']:<8}")

# Run detailed analysis for each dataset
for dataset in datasets_to_analyze:
    if dataset in all_results and "zeroshot_combined" in all_results[dataset]:
        per_organ_results = load_per_organ_results(dataset, "zeroshot_combined")
        
        if not per_organ_results:
            continue
        
        print(f"\n{'='*60}")
        print(f"DETAILED ANALYSIS FOR {dataset}")
        print('='*60)
        
        # Organ difficulty analysis
        organ_stats = create_organ_difficulty_ranking(per_organ_results, dataset)
        print_organ_difficulty_ranking(organ_stats, dataset, 'mean_presence')
        
        # Only show IoU ranking if we have meaningful IoU data
        if any(stats['mean_iou_bbox'] > 0 for stats in organ_stats.values()):
            print_organ_difficulty_ranking(organ_stats, dataset, 'mean_iou_bbox')
        
        # Model specialization analysis
        model_stats = create_model_specialization_analysis(per_organ_results, dataset)
        print_model_specialization(model_stats, dataset)
        
        # Find best model-organ combinations
        print(f"\n{dataset} - Standout Performances (>0.9 presence accuracy):")
        print("-" * 60)
        standout_count = 0
        for model, organ_data in per_organ_results.items():
            for organ, metrics in organ_data.items():
                if metrics['presence_acc'] >= 0.9:
                    print(f"  {model} → {organ}: {metrics['presence_acc']:.3f} presence accuracy")
                    standout_count += 1
        
        if standout_count == 0:
            print("  No model achieved >0.9 presence accuracy on any organ")
        
        # Find challenging organs (where no model does well)
        print(f"\n{dataset} - Challenging Organs (max presence accuracy <0.7):")
        print("-" * 60)
        challenging_count = 0
        for organ, stats in organ_stats.items():
            if stats['mean_presence'] < 0.7:
                # Find the best model for this organ
                best_score = 0
                best_model = "None"
                for model, organ_data in per_organ_results.items():
                    if organ in organ_data and organ_data[organ]['presence_acc'] > best_score:
                        best_score = organ_data[organ]['presence_acc']
                        best_model = model
                
                print(f"  {organ}: max {best_score:.3f} (by {best_model})")
                challenging_count += 1
        
        if challenging_count == 0:
            print("  All organs have at least one model achieving >0.7 presence accuracy")

print("\n" + "="*80)
print("SUMMARY ANALYSIS COMPLETE")
print("="*80)
print()
print("Key Insights:")
print("1. Heatmaps show per-organ performance across all models")
print("2. Organ difficulty rankings identify consistently hard/easy organs")
print("3. Model specialization analysis reveals generalists vs specialists")
print("4. Coefficient of Variation (CV) measures consistency across organs")
print("5. Standout performances highlight model-organ strengths")
print("6. Challenging organs may need specialized approaches")


PER-ORGAN SUMMARY ANALYSIS


NameError: name 'datasets_to_analyze' is not defined